# Lab 13: The fact store

*Project LANTERN, Tool Tutorial Part 2. Classroom sample: Project Wren (`deal_a`) and its rival Project Heron (`deal_b`). Fictional, for DAMG 7245.*

In [ ]:
import os, sys, json
from pathlib import Path
ROOT = Path.cwd()
while not (ROOT / "params.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
from src.llm import client as llm
from src import schemas, prompts, sample_data as sd
print("repository:", ROOT, "| LLM mode:", llm.MODE)

In [ ]:
import duckdb
Path("data/work").mkdir(parents=True, exist_ok=True)
con = duckdb.connect("data/work/facts.duckdb")
con.execute("DROP INDEX IF EXISTS f_lookup")   # rerun-safe: the index first, then the table
con.execute("DROP TABLE IF EXISTS facts")

From the slide, the fact table:

```sql
CREATE TABLE facts (
  fact_id     VARCHAR PRIMARY KEY, -- sha1 of the key
  deal_id     VARCHAR NOT NULL,    -- 'public' | deal
  entity_id   VARCHAR NOT NULL,    -- CIK or deal id
  concept     VARCHAR,             -- us-gaap:Revenues
  metric      VARCHAR NOT NULL,    -- vocab: revenue
  value       DOUBLE  NOT NULL,
  unit        VARCHAR NOT NULL,    -- USD, shares, pure
  period_type VARCHAR,             -- duration|instant
  period_start DATE, period_end DATE,
  decimals    INTEGER,
  tier        TINYINT NOT NULL,    -- 1..3 only here
  source_type VARCHAR NOT NULL,
  doc_id      VARCHAR NOT NULL,
  citation    JSON    NOT NULL,    -- kind + locator
  extractor   VARCHAR,
  lineage     JSON                 -- parents, sha256
);
CREATE INDEX f_lookup ON facts(deal_id, metric,
                               period_end);
```

In [ ]:
# Run the slide's DDL, one statement at a time
DDL = """CREATE TABLE facts (
  fact_id     VARCHAR PRIMARY KEY, -- sha1 of the key
  deal_id     VARCHAR NOT NULL,    -- 'public' | deal
  entity_id   VARCHAR NOT NULL,    -- CIK or deal id
  concept     VARCHAR,             -- us-gaap:Revenues
  metric      VARCHAR NOT NULL,    -- vocab: revenue
  value       DOUBLE  NOT NULL,
  unit        VARCHAR NOT NULL,    -- USD, shares, pure
  period_type VARCHAR,             -- duration|instant
  period_start DATE, period_end DATE,
  decimals    INTEGER,
  tier        TINYINT NOT NULL,    -- 1..3 only here
  source_type VARCHAR NOT NULL,
  doc_id      VARCHAR NOT NULL,
  citation    JSON    NOT NULL,    -- kind + locator
  extractor   VARCHAR,
  lineage     JSON                 -- parents, sha256
);
CREATE INDEX f_lookup ON facts(deal_id, metric,
                               period_end);"""
for stmt in [s for s in DDL.split(";") if s.strip()]:
    con.execute(stmt)

In [ ]:
# From the slide: Seller financials from xlsx: tier-3 facts with cell citations
from openpyxl import load_workbook

def xlsx_facts(path, sheet, row_map, deal_id, doc_id):
    wb = load_workbook(path, data_only=True,
                       read_only=True)   # values only
    rows = wb[sheet].iter_rows()
    header = [c.value for c in next(rows)]
    for row in rows:
        label = row[0].value
        if label not in row_map:         # vocab mapping
            continue
        for j, cell in enumerate(row[1:], start=1):
            ref = f"{sheet}!{cell.coordinate}"
            if cell.value is None:       # fail loudly
                raise ValueError(f"{ref}: empty or "
                                 "uncached formula")
            yield {"deal_id": deal_id, "tier": 3,
                   "metric": row_map[label],
                   "period": header[j],
                   "value": float(cell.value),
                   "citation": {"kind": "sheet_cell",
                                "doc_id": doc_id,
                                "cell": ref}}

In [ ]:
row_map = {"Revenue": "revenue", "Adjusted EBITDA": "adjusted_ebitda"}
raw = list(xlsx_facts("data/dataroom/deal_a/raw/financials.xlsx", "P&L", row_map, "deal_a", sd.XLSX_DOC))
print(raw[2])
# The sheet is in USD millions: scale before facts meet the linter, which reads "$41.2 million" as 41.2e6
facts = sd.seller_facts()
assert all(any(f["citation"]["cell"] == r["citation"]["cell"] and abs(f["value"] - r["value"] * 1e6) < 1 for f in facts) for r in raw)
print(len(facts), "seller facts, scaled; e.g.", [(f["metric"], f["period"], f["value"]) for f in facts if f["metric"] == "revenue"])

In [ ]:
from datetime import date
def fiscal_bounds(label, fye_month=6):          # Wren's fiscal year ends June 30
    y = int(label[2:]); return date(y - 1, fye_month + 1, 1).isoformat(), date(y, fye_month, 30).isoformat()
rows = []
for f in facts:
    start, end = fiscal_bounds(f["period"])
    rows.append((f["fact_id"], "deal_a", "deal_a", None, f["metric"], f["value"], f["unit"], "duration", start, end, None,
                 f["tier"], "dataroom", f["citation"]["doc_id"], json.dumps(f["citation"]), "openpyxl", json.dumps({"parents": f["citation"].get("parents", [])})))
con.executemany("INSERT INTO facts VALUES (" + ",".join(["?"] * 17) + ")", rows)
con.execute("SELECT metric, period_end, value, tier FROM facts WHERE deal_id = 'deal_a' ORDER BY metric, period_end").df()

In [ ]:
# From the slide: Citations for XBRL facts: a bbox if you can, an element if you cannot
def xbrl_citation(fact, label_map_hits, doc_id):
    """Prefer the PDF cell CS1 Part 11 matched."""
    hit = label_map_hits.get((fact["concept"],
                              fact["period_end"]))
    if hit:                        # rendered + matched
        return {"kind": "pdf_bbox", "doc_id": doc_id,
                "page": hit["page"], "bbox": hit["bbox"],
                "block_id": hit["block_id"]}
    if fact.get("element_id"):     # ix:nonFraction id
        return {"kind": "ixbrl_element", "doc_id": doc_id,
                "element_id": fact["element_id"]}
    return {"kind": "ixbrl_context", "doc_id": doc_id,
            "concept": fact["concept"],
            "context_ref": fact["context_ref"],
            "unit_ref": fact.get("unit_ref")}

In [ ]:
fact = {"concept": "us-gaap:Revenues", "period_end": "2025-12-31", "context_ref": "c-FY2025", "unit_ref": "usd"}
print(xbrl_citation(fact, {("us-gaap:Revenues", "2025-12-31"): {"page": 41, "bbox": [72, 410, 540, 424], "block_id": "p0041_b007"}}, "0000000001-25-000101"))
print(xbrl_citation(fact, {}, "0000000001-25-000101"))   # no CS1 match and no element id -> ixbrl_context

**Exercise 13.** Add the KPIs sheet (NRR, gross logo churn) with the right scale for each row. Which KPI would break if you scaled the whole sheet by 1e6?